In [16]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("../data/fraud_oracle_with_telematics.csv")
print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())

Shape: (15420, 43)

Columns:
 ['Month', 'WeekOfMonth', 'DayOfWeek', 'Make', 'AccidentArea', 'DayOfWeekClaimed', 'MonthClaimed', 'WeekOfMonthClaimed', 'Sex', 'MaritalStatus', 'Age', 'Fault', 'PolicyType', 'VehicleCategory', 'VehiclePrice', 'FraudFound_P', 'PolicyNumber', 'RepNumber', 'Deductible', 'DriverRating', 'Days_Policy_Accident', 'Days_Policy_Claim', 'PastNumberOfClaims', 'AgeOfVehicle', 'AgeOfPolicyHolder', 'PoliceReportFiled', 'WitnessPresent', 'AgentType', 'NumberOfSuppliments', 'AddressChange_Claim', 'NumberOfCars', 'Year', 'BasePolicy', 'avg_speed_kmph', 'max_speed_kmph', 'hard_brakes_per_trip', 'rapid_acceleration_events', 'trip_duration_minutes', 'distance_km', 'night_driving_ratio', 'urban_driving_ratio', 'harsh_cornering_events', 'idle_time_minutes']


In [ ]:
# Step 1: Normalize column names and text values
df.columns = df.columns.str.strip().str.lower()

for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = df[col].astype(str).str.strip().str.lower()

Columns normalized to lowercase
Object columns to encode: ['month', 'dayofweek', 'make', 'accidentarea', 'dayofweekclaimed', 'monthclaimed', 'sex', 'maritalstatus', 'fault', 'policytype', 'vehiclecategory', 'vehicleprice', 'days_policy_accident', 'days_policy_claim', 'pastnumberofclaims', 'ageofvehicle', 'ageofpolicyholder', 'policereportfiled', 'witnesspresent', 'agenttype', 'numberofsuppliments', 'addresschange_claim', 'numberofcars', 'basepolicy']


In [18]:
# Step 2: Handle missing values
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
df[num_cols] = df[num_cols].apply(lambda x: x.fillna(x.median()))

cat_cols = df.select_dtypes(include=['object', 'string']).columns
for col in cat_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print(f"Missing values after fill: {df.isnull().sum().sum()}")

Missing values after fill: 0


In [ ]:
# Step 3: Map range-based columns to numeric values
range_mappings = {
    'days_policy_claim': {'none': 0, '1 to 7': 3, '8 to 15': 10, '15 to 30': 20, 'more than 30': 40},
    'days_policy_accident': {'none': 0, '1 to 7': 3, '8 to 15': 10, '15 to 30': 20, 'more than 30': 40},
    'pastnumberofclaims': {'none': 0, '1': 1, '2 to 4': 3, 'more than 4': 5},
    'numberofsuppliments': {'none': 0, '1 to 2': 1.5, '3 to 5': 4, 'more than 5': 6}
}

for col, mapping in range_mappings.items():
    if col in df.columns:
        df[col] = df[col].replace(mapping)
        df[col] = pd.to_numeric(df[col])

✓ days_policy_claim -> numeric
✓ days_policy_accident -> numeric
✓ pastnumberofclaims -> numeric
✓ numberofsuppliments -> numeric
All range columns converted to numeric


In [ ]:
# Step 4: Create derived telematics features
if 'max_speed_kmph' in df.columns:
    df['speeding_risk'] = (df['max_speed_kmph'] > 120).astype(int)
    df['speed_volatility'] = (df['max_speed_kmph'] - df.get('avg_speed_kmph', df['max_speed_kmph'])).abs()

if 'hard_brakes_per_trip' in df.columns:
    df['harsh_braking_risk'] = (df['hard_brakes_per_trip'] > 5).astype(int)

if 'rapid_acceleration_events' in df.columns:
    df['harsh_acceleration_risk'] = (df['rapid_acceleration_events'] > 5).astype(int)

if 'harsh_cornering_events' in df.columns:
    df['harsh_cornering_risk'] = (df['harsh_cornering_events'] > 5).astype(int)

behavior_cols = ['hard_brakes_per_trip', 'rapid_acceleration_events', 'harsh_cornering_events']
if all(col in df.columns for col in behavior_cols):
    df['harsh_driving_index'] = (df['hard_brakes_per_trip'] + df['rapid_acceleration_events'] + df['harsh_cornering_events']) / 3

if 'night_driving_ratio' in df.columns:
    df['high_night_driving'] = (df['night_driving_ratio'] > 0.5).astype(int)

if 'urban_driving_ratio' in df.columns:
    df['high_urban_driving'] = (df['urban_driving_ratio'] > 0.7).astype(int)


=== TELEMATICS FEATURE ENGINEERING ===
✓ Telematics features created


In [ ]:
# Step 5: Feature Engineering - Claim Features
if 'days_policy_claim' in df.columns:
    df['fast_claim'] = (df['days_policy_claim'] < 7).astype(int)

if 'pastnumberofclaims' in df.columns:
    df['high_claim_history'] = (df['pastnumberofclaims'] > 2).astype(int)

if 'fast_claim' in df.columns and 'harsh_driving_index' in df.columns:
    df['claim_driving_risk'] = df['fast_claim'] * df['harsh_driving_index']


=== CLAIM FEATURE ENGINEERING ===
✓ Claim features created


In [ ]:
# Step 5a: Remove ID leakage columns
id_cols = ['policynumber', 'repnumber']
for col in id_cols:
    if col in df.columns:
        df = df.drop(col, axis=1)

# Step 5b: Convert binary yes/no columns to numeric
binary_yes_no = ['policereportfiled', 'witnesspresent']
for col in binary_yes_no:
    if col in df.columns:
        df[col] = df[col].map({'yes': 1, 'no': 0})

# Step 5c: Label-encode categorical columns
cat_for_model = ['policytype', 'vehiclecategory', 'vehicleprice', 'ageofvehicle', 'ageofpolicyholder', 'numberofcars']
cat_present = [col for col in cat_for_model if col in df.columns]

from sklearn.preprocessing import LabelEncoder
for col in cat_present:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))


=== REMOVING ID LEAKAGE COLUMNS ===
✓ Dropped policynumber
✓ Dropped repnumber

=== CONVERTING BINARY COLUMNS ===
✓ policereportfiled → numeric (yes=1, no=0)
✓ witnesspresent → numeric (yes=1, no=0)

=== CATEGORICAL COLUMNS FOR CATBOOST ===
✓ Keeping as categorical: ['policytype', 'vehiclecategory', 'vehicleprice', 'ageofvehicle', 'ageofpolicyholder', 'numberofcars']
✓ Label encoded 6 categorical columns for CatBoost


In [ ]:
# Step 6: Identify numeric columns for scaling
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()


=== NUMERIC FEATURES FOR SCALING ===
Found 40 numeric columns
Columns: ['weekofmonth', 'weekofmonthclaimed', 'age', 'policytype', 'vehiclecategory', 'vehicleprice', 'fraudfound_p', 'deductible', 'driverrating', 'days_policy_accident', 'days_policy_claim', 'pastnumberofclaims', 'ageofvehicle', 'ageofpolicyholder', 'policereportfiled']


In [ ]:
# Step 7: Standard Scaling for numeric features
scaler = StandardScaler()
exclude_from_scaling = ['fraudfound_p', 'year']
scale_cols = [col for col in numeric_cols if col not in exclude_from_scaling]

df[scale_cols] = scaler.fit_transform(df[scale_cols])


=== SCALING NUMERIC FEATURES ===
✓ Scaled 38 numeric features
✓ Excluded from scaling: ['fraudfound_p', 'year']


In [ ]:
# Step 8: Final validation
print("=== PREPROCESSING CHECK ===")
print(f"Shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Fraud rate: {(df['fraudfound_p'] == 1).sum() / len(df):.2%}")


=== FINAL PREPROCESSING CHECK ===
Shape: (15420, 52)
Missing values: 0

Data types:
float64    38
str        12
int64       2
Name: count, dtype: int64

Target variable (fraudfound_p):
Fraud rate: 5.99%

Numeric (scaled): 38 columns
Categorical (label-encoded for CatBoost): ~6 columns

✓ Ready for CatBoost training


In [ ]:
# Display sample
df.head()


Sample of processed data:
  month  weekofmonth  dayofweek    make accidentarea dayofweekclaimed  \
0   dec     1.717545  wednesday   honda        urban          tuesday   
1   jan     0.164199  wednesday   honda        urban           monday   
2   oct     1.717545     friday   honda        urban         thursday   
3   jun    -0.612473   saturday  toyota        rural           friday   
4   jan     1.717545     monday   honda        urban          tuesday   

  monthclaimed  weekofmonthclaimed     sex maritalstatus  ...  \
0          jan           -1.345408  female        single  ...   
1          jan            1.037295    male        single  ...   
2          nov           -0.551174    male       married  ...   
3          jul           -1.345408    male       married  ...   
4          feb           -0.551174  female        single  ...   

   speed_volatility harsh_braking_risk  harsh_acceleration_risk  \
0          0.725376          -0.301255                -0.524215   
1        

In [ ]:
# Save processed data
df.to_csv("../data/processed.csv", index=False)
print("✓ Processed data saved to: data/processed.csv")


PREPROCESSING COMPLETE - CATBOOST READY
✓ Processed file saved: ../data/processed.csv
✓ Final shape: (15420, 52)
✓ ID columns removed: policynumber, repnumber
✓ Binary yes/no columns converted to 1/0: policereportfiled, witnesspresent
✓ Categorical columns label-encoded for CatBoost: ~6
✓ Numeric features scaled: 38
✓ Fraud distribution: 923 frauds (5.99%)


In [ ]:
# Verify target variable
print("Target distribution:")
print(df['fraudfound_p'].value_counts())


Target variable check:
fraudfound_p in columns: True
fraudfound_p dtype: int64
fraudfound_p unique: [0 1]
fraudfound_p value counts:
fraudfound_p
0    14497
1      923
Name: count, dtype: int64


In [ ]:
# Final categorical encoding
remaining_str_cols = df.select_dtypes(include='object').columns.tolist()

if remaining_str_cols:
    from sklearn.preprocessing import LabelEncoder
    for col in remaining_str_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
    df.to_csv("../data/processed.csv", index=False)


=== FINAL CATEGORICAL CHECK ===
Remaining string columns: ['month', 'dayofweek', 'make', 'accidentarea', 'dayofweekclaimed', 'monthclaimed', 'sex', 'maritalstatus', 'fault', 'agenttype', 'addresschange_claim', 'basepolicy']
✓ Label encoded month
✓ Label encoded dayofweek
✓ Label encoded make
✓ Label encoded accidentarea
✓ Label encoded dayofweekclaimed
✓ Label encoded monthclaimed
✓ Label encoded sex
✓ Label encoded maritalstatus
✓ Label encoded fault
✓ Label encoded agenttype
✓ Label encoded addresschange_claim
✓ Label encoded basepolicy

✓ Re-saved processed.csv with all columns properly encoded

Final shape: (15420, 52)
Final data types:
float64    38
int64      14
Name: count, dtype: int64


In [ ]:
# Final summary
print("\n" + "=" * 80)
print("✓ PREPROCESSING COMPLETE - CATBOOST READY")
print("=" * 80)
print(f"Shape: {df.shape} | Fraud rate: {(df['fraudfound_p'] == 1).sum() / len(df):.2%}")
print(f"Features: 38 numeric (scaled) + 12 categorical (encoded)")
print(f"Output: data/processed.csv")
print("=" * 80)


PREPROCESSING COMPLETE - READY FOR CATBOOST

FIXES APPLIED:
  Removed ID leakage: policynumber, repnumber
  ✓ Converted binary yes/no → 1/0: policereportfiled, witnesspresent
  ✓ Label-encoded all 12 categorical columns for CatBoost
  ✓ Created 8+ telematics features from driving behavior data
  ✓ Created claim risk features
  ✓ Scaled 38 numeric features (mean=0, std=1)

DATASET STATS:
  • Shape: (15420, 52)
  • Target (fraudfound_p): 923 fraud cases (5.99%)
  • Numeric features (scaled): 38
  • Categorical features (label-encoded): 12
  • Missing values: 0

OUTPUT: data/processed.csv
